# 🤖 Module 1: LLM API Foundations
**Agentic AI Engineering Program**

An agent, in this program, is software that uses a language model to get something done: read
input, decide what to do, take an action, decide again. However elaborate that behavior looks
from outside, all of it is built from one repeated move underneath: send the model a request,
get text back. That request has one concrete shape, an HTTP call to `POST /v1/messages`,
Anthropic's Messages API, and everything you will build later (tool use, streaming, memory,
sub-agents, evals) is a variation on that same call, not a separate mechanism. Get comfortable
with the request itself first, and the rest of the program stops looking like magic and starts
looking like plumbing you understand.

This module builds that comfort, roughly in the order you will need it. The model itself
remembers nothing between calls, so the first thing you will see is what a *conversation*
actually is: you resending the whole history yourself, every turn. From there: how to control
what the model generates, how to keep a growing conversation inside its context window, how to
stream a response as it arrives, how to make the model return structured data instead of prose,
and how to stop paying to reprocess the same tokens on every turn.


| Section | Content |
|---|---|
| 1.0 | Setup: client, model, offline mode |
| 1.1 | Messages API: roles and conversation state |
| 1.2 | Sampling and decoding controls |
| 1.3 | Tokens and the context window |
| 1.4 | Streaming |
| 1.5 | Structured output |
| 1.6 | Prompt caching |

Each section runs the code first. You send a small request, look at what came back, and the
lesson that follows explains what you just saw. Pitfalls are demonstrated rather than
announced, usually as a cell written to fail on purpose. Each section then closes with three
exercises that build on one another: one to confirm you followed, one where most of the code
is already written, one you write yourself. Anything genuinely advanced is marked optional
and kept off the main path.

Lesson cells run with or without an API key: without one they print the exact JSON payload that
*would* be sent. Exercises never need a key: they run against recorded API objects and pure logic,
so a solution is deterministically right or wrong.

---
## 1.0 Setup: client, model, offline mode

Everything in this module goes through one client object and one model id. Getting both right
takes five minutes and removes a whole class of confusing failures later, so we do it first and
then stop thinking about it. This section also puts in place the two conveniences the rest of the
notebook leans on: lesson code that runs whether or not you have an API key, and a `check()`
helper that tells you when an exercise is right.

In [ ]:
# ── Install (run once) ───────────────────────────────────────
# pip install anthropic pydantic

import json, os, time, math, random, re
from typing import Any, Callable

# ── Credentials come from the environment, never from the source ──
try:
    import anthropic
    CLIENT = anthropic.Anthropic()   # ANTHROPIC_API_KEY | ANTHROPIC_AUTH_TOKEN | `ant auth login`
except Exception as exc:
    CLIENT = None
    print(f"offline mode ({type(exc).__name__}): {exc}")

LIVE = CLIENT is not None
print("LIVE =", LIVE)


### Lesson: The client is a keyring, not a key

Do not picture `Anthropic()` as needing one key. It behaves more like a keyring: the SDK tries an
environment variable, then an auth token, then a saved CLI login, then workload identity
federation, in that order, before giving up. An unset `ANTHROPIC_API_KEY` therefore does not prove
that nothing will authenticate. It may just mean the key is in the second or third pocket.

Which is why the constructor above takes no arguments. Hardcoding `api_key="sk-ant-..."` into
source is the classic mistake, and it is not only a secrets problem: it short-circuits the whole
resolution chain, so the code stops working anywhere the credential arrives by another route.

`LIVE` records which branch you landed in. Everything below works either way.


In [ ]:
# ── Model ids are complete as-is. Never append a date suffix ─
MODEL       = "claude-opus-5"      # 1M context, 128K max output, $5 / $25 per 1M tokens
CHEAP_MODEL = "claude-haiku-4-5"   # 200K context, $1 / $5, for cheap sub-tasks and classification

WRONG = "claude-opus-5-20260101"   # a 404, not an auth error
print("using:", MODEL, "| cheap:", CHEAP_MODEL)


### Lesson: A model id is a name you look up

It is not a password you half-remember and retype. Current ids are complete exactly as they are,
and bolting on a date suffix copied from an old snippet, `claude-opus-5-20260101` and friends,
returns a 404.

That particular 404 is worth recognising on sight, because it gets misdiagnosed as an
authentication problem, after which people spend an hour going through their credentials. A wrong
key and a wrong model id fail differently, and the model id is the cheaper of the two to rule out
first.


In [ ]:
# ── The three helpers the rest of this notebook runs on ──────
def call(**kwargs):
    "Send the request when a client exists; otherwise print the payload that would be sent."
    if not LIVE:
        print("[offline] POST /v1/messages")
        print(json.dumps(kwargs, indent=2, default=str)[:900])
        return None
    return CLIENT.messages.create(**kwargs)


def text_of(response) -> str:
    "response.content is a LIST of blocks (text, thinking, tool_use...). Never assume [0] is text."
    if response is None:
        return ""
    return "".join(b.text for b in response.content if b.type == "text")


def check(label, got, want):
    "Exercise feedback: compares the RESULT, and shows both sides when they differ."
    if callable(got):            # a lambda: an unfinished exercise reports, it does not crash
        try:
            got = got()
        except Exception as exc:
            got = f"<{type(exc).__name__}: {exc}>"
    ok = got == want
    print(("PASS  " if ok else "FAIL  ") + label)
    if not ok:
        print("   got :", repr(got))
        print("   want:", repr(want))
    return ok


# Demo calls in this notebook use a small max_tokens to keep the drill cheap.
# Production defaults: ~16000 non-streaming, ~64000 streaming (see 1.2 and 1.4).
DEMO_MAX_TOKENS = 300

check("this is what a passing exercise looks like", 2 + 2, 4)
check("and this is what a failing one looks like", 2 + 2, 5)


---
## 1.1 Messages API: roles and conversation state

This is the call everything else is built on. You send a list of messages, you get text back, and
between two calls the model remembers nothing at all. That last part is not a limitation to work
around, it is the design, and once it clicks an entire family of agent bugs stops being
mysterious. By the end of this section you can assemble a transcript the API will accept, and
continue it without throwing away anything the model produced.

In [ ]:
# ── Your first call ──────────────────────────────────────────
# The entire Messages API, in one request. Run it.
resp = call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    messages=[{"role": "user", "content": "What is an agent loop? One sentence."}],
)
print(text_of(resp))


### Lesson: The request carries the whole conversation

You sent a list called `messages` and got text back. That list is not a handle pointing at a
session stored on Anthropic's side. It *is* the conversation, in full, and you resend it every
time.

Picture a brilliant consultant with no memory of any call before this one. Every time you ring,
you fax over the entire case file first, or they cannot say a word. Nothing gets forgotten
between your calls, because there is nothing on their end to forget. Each request is answered
purely on what came attached to it.

Run the next cell and watch what that costs you when you ignore it.


In [ ]:
# ── Two separate calls: the second one cannot possibly know ──
first  = [{"role": "user", "content": "My name is Adrien."}]
second = [{"role": "user", "content": "What is my name?"}]      # the name is nowhere in here

call(model=MODEL, max_tokens=DEMO_MAX_TOKENS, messages=first)
call(model=MODEL, max_tokens=DEMO_MAX_TOKENS, messages=second)  # a guess, or a polite refusal

# ── The fix is not a setting or a session id. You resend. ────
carried = [
    {"role": "user",      "content": "My name is Adrien."},
    {"role": "assistant", "content": "Nice to meet you, Adrien."},
    {"role": "user",      "content": "What is my name?"},        # answerable now
]
call(model=MODEL, max_tokens=DEMO_MAX_TOKENS, messages=carried)

# Offline, `call` prints the payload instead of sending it. Compare the two
# payloads above: the name is simply not present in the second one.


### Lesson: Three roles, three levels of trust

Every entry in `messages` carries a role, and the role is not decoration. `system` is operator
instruction, and it sits outside the conversation entirely, as a top-level parameter of the
request. `user` is input. `assistant` is what the model produced on an earlier turn.

That distinction starts earning its keep the moment your agent handles text it did not write: a
retrieved document, a scraped page, the output of a tool. All of it belongs in `user` content.
Move it into `system` and you have promoted whatever it happens to say to the status of an
instruction from you, which is the whole mechanism behind prompt injection.


In [ ]:
# ── system is a top-level parameter, not an entry in messages ─
call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    system="You are a terse assistant. Answer in one sentence.",
    messages=[{"role": "user", "content": "What is an agent loop?"}],
)

# ── Two rules the API enforces on the messages list ──────────
# 1. messages[0] must have role "user"
illegal = [{"role": "assistant", "content": "Hi there!"}]        # 400 from the API

# 2. consecutive same-role entries are legal, and merge into one turn server-side
legal_merge = [
    {"role": "user", "content": "First thought."},
    {"role": "user", "content": "Second thought, same turn."},
    {"role": "assistant", "content": "Got both."},
]


### Lesson: `content` is a list, even when it looks like a string

`"content": "Summarise this log."` is shorthand. On the wire it becomes
`[{"type": "text", "text": "Summarise this log."}]`, and that longer form is what comes back to
you. A response is an envelope holding separate items, not one paragraph: a `text` block, maybe
a `thinking` block ahead of it, maybe a `tool_use` block asking you to go look something up.

Which is why `response.content[0].text` is a trap. It works on your first toy call, then breaks
the day you switch on thinking or tools, because index 0 is no longer the text block. The next
cell breaks it on purpose.


In [ ]:
# ── A response with thinking on: index 0 is NOT the text block ─
class Block:
    "Stand-in for anthropic.types.*Block: attribute access, .type discriminator."
    def __init__(self, **kw):
        self.__dict__.update(kw)

recorded = type("Response", (), {})()
recorded.content = [
    Block(type="thinking", thinking="They want a definition.", signature="sig_abc"),
    Block(type="text",     text="A loop that calls the model until it stops asking for tools."),
]

try:
    print(recorded.content[0].text)      # the thinking block has no .text
except AttributeError as exc:
    print("AttributeError:", exc)

print(text_of(recorded))                 # filters on block.type instead (defined in 1.0)


### Lesson: Append the blocks, not the string

To continue a conversation you append the assistant's turn to `messages` and resend the lot.
*What* you append matters more than it looks:

```python
messages.append({"role": "assistant", "content": response.content[0].text})   # wrong
```

This single line is the most common bug in hand-rolled agent loops. It keeps the prose and drops
everything else. With thinking enabled, the model loses its own reasoning from the turn before.
With tools, it drops the `tool_use` block, so the `tool_result` you send next refers to an id
that no longer appears anywhere in the transcript, and the API rejects it with something that
reads like a schema error.

Append `response.content` whole and none of that can happen.


In [ ]:
# ── A minimal, correct conversation manager ──────────────────
class Conversation:
    "Stateless API plus a client-side transcript. Appends BLOCKS, never a bare string."

    def __init__(self, model=MODEL, system=None):
        self.model    = model
        self.system   = system
        self.messages = []

    def send(self, user_message, **kwargs):
        self.messages.append({"role": "user", "content": user_message})
        response = call(
            model=self.model,
            max_tokens=kwargs.pop("max_tokens", DEMO_MAX_TOKENS),
            system=self.system,
            messages=self.messages,
            **kwargs,
        )
        if response is not None:
            self.messages.append({"role": "assistant", "content": response.content})
        return response


chat = Conversation(system="You are a helpful assistant. Be brief.")
chat.send("My name is Adrien.")
chat.send("What is my name?")            # only answerable because turn 1 gets resent
print("transcript:", [m["role"] for m in chat.messages])
# Offline there are no assistant turns: `call` returned None, so nothing was appended.


### 🏋️ Exercises 1.1


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 (warm-up) — Which request is legal?
# ════════════════════════════════════════════════════════
# Exactly one of the three lists below is a legal `messages` value.
# Set ANSWER to its name and run the cell. Nothing to implement.

a = [{"role": "assistant", "content": "Hi! How can I help?"},
     {"role": "user",      "content": "Summarise this log."}]

b = [{"role": "user",      "content": "Summarise this log."},
     {"role": "user",      "content": "Focus on the timeouts."},
     {"role": "assistant", "content": "Timeouts dominate: 412 lines out of 500."}]

c = [{"role": "system",    "content": "You are a log analyst."},
     {"role": "user",      "content": "Summarise this log."}]

ANSWER = ""        # "a", "b" or "c"

check("legal messages list", ANSWER, "b")

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# ANSWER = "b"
#   a opens on an assistant turn, and messages[0] must be "user".
#   c puts the operator prompt inside messages, but `system` is a top-level
#     parameter of the request, sitting next to `model` and `messages`.
#   b is fine: two consecutive user entries are legal, the API merges them.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 (fill in the gaps) — Continue the conversation
# ════════════════════════════════════════════════════════
# `response` mimics what the SDK hands back: .content is a LIST of blocks.
# Fill the two marked lines so the assistant turn is appended without losing
# a block, and the user's follow-up lands after it.

response = type("Response", (), {})()
response.content = [
    Block(type="thinking", thinking="They want the weather.", signature="sig_abc"),
    Block(type="text",     text="Let me look that up."),
]

def continue_conversation(messages, response, follow_up):
    # The shape is already right. Replace the two placeholder values.
    messages.append({"role": "assistant", "content": []})     # (1) every block, not just the text
    messages.append({"role": "user", "content": ""})          # (2) the follow-up
    return messages

out = continue_conversation([{"role": "user", "content": "Weather in Paris?"}],
                            response, "And tomorrow?")
check("roles, in order",  [m["role"] for m in out], ["user", "assistant", "user"])
check("no block dropped", len(out[1]["content"]), 2)
check("follow-up text",   out[2]["content"], "And tomorrow?")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def continue_conversation(messages, response, follow_up):
#     messages.append({"role": "assistant", "content": response.content})
#     messages.append({"role": "user", "content": follow_up})
#     return messages


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Build a legal transcript from raw events
# ════════════════════════════════════════════════════════
# `events` is a list of (role, text) tuples in arrival order. Return a
# `messages` list obeying the two rules from the lesson:
#   • it cannot open on an assistant turn, so drop any leading ones
#   • consecutive same-role events are one turn, join their text with "\n"

def build_transcript(events):
    pass  # to complete

events = [
    ("assistant", "stray opening"),
    ("user",      "First thought."),
    ("user",      "Second thought."),
    ("assistant", "Got both."),
    ("user",      "Continue."),
]
check("transcript", build_transcript(events), [
    {"role": "user",      "content": "First thought.\nSecond thought."},
    {"role": "assistant", "content": "Got both."},
    {"role": "user",      "content": "Continue."},
])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def build_transcript(events):
#     msgs = []
#     for role, text in events:
#         if not msgs and role != "user":
#             continue
#         if msgs and msgs[-1]["role"] == role:
#             msgs[-1]["content"] += "\n" + text
#         else:
#             msgs.append({"role": role, "content": text})
#     return msgs


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Changing the operator instruction mid-conversation is a separate feature,
# gated by model. On Opus 5 / Opus 4.8 / Fable 5 you append a {"role": "system"}
# MESSAGE rather than editing top-level `system`, which would otherwise force
# every cached token ahead of it to be reprocessed (1.6 covers why). Placement
# rules are strict. A system entry is legal only if
#   (a) it is not messages[0]
#   (b) the entry before it has role "user"
#   (c) it is last, or the entry after it has role "assistant"
# Return the indices of the entries that break a rule.

def validate_system_placement(messages):
    pass  # to complete

conv = [
    {"role": "system",    "content": "boot"},           # illegal: first
    {"role": "user",      "content": "hi"},
    {"role": "system",    "content": "terse mode"},     # legal: after user, before assistant
    {"role": "assistant", "content": "ok"},
    {"role": "system",    "content": "verbose mode"},   # illegal: follows an assistant turn
    {"role": "user",      "content": "go"},
    {"role": "system",    "content": "final"},          # legal: last entry
]
check("offending indices", validate_system_placement(conv), [0, 4])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def validate_system_placement(messages):
#     bad = []
#     for i, m in enumerate(messages):
#         if m["role"] != "system":
#             continue
#         if i == 0 or messages[i - 1]["role"] != "user":
#             bad.append(i)
#         elif i < len(messages) - 1 and messages[i + 1]["role"] != "assistant":
#             bad.append(i)
#     return bad


---
## 1.2 Sampling and decoding controls

A model does not choose a next token. It scores every token it knows, and something downstream
draws one from that spread. This section is about what you can and cannot control in that step.
On current models the honest answer is: less than you would expect, and none of the parameters a
tutorial written before 2026 will tell you to set. You also learn to read `stop_reason`, the field
that tells you whether the answer you just received is even finished.

In [ ]:
# ── The request you would actually write today ───────────────
resp = call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    thinking={"type": "adaptive", "display": "summarized"},   # default display is "omitted"
    output_config={"effort": "low"},                          # low | medium | high | xhigh | max
    messages=[{"role": "user", "content": "Classify: 'refund not received'. One word."}],
)
print(text_of(resp))

# Notice what is NOT in there: no temperature, no top_p, no top_k.


### Lesson: The model emits a distribution, not a token

At every step the model produces a score for each of roughly 200k possible next tokens. Decoding
parameters reshape that spread of scores before one token is drawn from it.

Rank a restaurant's whole menu from favourite to least favourite, then decide how adventurous you
feel. Near `temperature` 0 you order dish #1 every night: safe, and identical forever. Turn it up
and the dishes you ranked low start getting picked. `top_p` is adventurous differently: rather
than reshuffling your preferences, it keeps only the dishes that together account for 80% of what
you would plausibly enjoy, and chooses inside that shortlist.

The next cell does the reshaping in pure Python, no API involved.


In [ ]:
# ── The mechanism, in pure Python (no API needed) ────────────
def softmax(logits: list[float], temperature: float = 1.0) -> list[float]:
    "Temperature scales the logits BEFORE normalising: low T sharpens, high T flattens."
    scaled = [x / temperature for x in logits]
    m = max(scaled)                                  # subtract max for numerical stability
    exps = [math.exp(x - m) for x in scaled]
    total = sum(exps)
    return [e / total for e in exps]


logits = [3.2, 2.9, 1.1, 0.4, -0.8]                  # 5 candidate tokens
for t in (0.2, 1.0, 2.0):
    probs = softmax(logits, t)
    print(f"T={t:<4} -> " + " ".join(f"{p:.3f}" for p in probs))

# T=0.2 puts nearly all the mass on the top token: greedy, repetitive.
# T=2.0 hands real probability to tokens the model rated unlikely: creative, off-task.


### Lesson: The knobs are gone, and `effort` replaced them

Here is the part that dates every agent tutorial written before 2026. On Claude Opus 5, Opus 4.8,
Opus 4.7, Sonnet 5 and Fable 5, `temperature`, `top_p` and `top_k` have been **removed**. Sending
any of them returns a 400. They still work on Opus 4.6, Sonnet 4.6 and older.

So "set `temperature=0` for agents" is not merely dated, it now breaks the request. And it never
delivered what people wanted anyway: batching, GPU non-associativity and routing make identical
requests diverge regardless. Code that relied on "temperature 0 means reproducible" was relying on
a bug in its own mental model.

What you tune instead is `output_config.effort`: how hard the model thinks, not how randomly it
answers.


In [ ]:
# ── What a current model accepts, and what it rejects ────────
rejected = {"temperature": 0.0, "top_p": 0.9, "top_k": 40}   # 400 on Opus 5 / 4.8 / 4.7, Sonnet 5, Fable 5
accepted = {"thinking": {"type": "adaptive"},                # on by default on Opus 5
            "output_config": {"effort": "xhigh"}}

# Effort by workload, the practical mapping:
#   low    : classification, routing, sub-agents, cheap extraction
#   medium : routine transformation, summarisation
#   high   : default; intelligence-sensitive work
#   xhigh  : coding and long-horizon agentic tasks (best default on Opus 5 / Sonnet 5)
#   max    : correctness matters more than cost
print(json.dumps(accepted, indent=1))


### Lesson: `max_tokens` is a guillotine, not a hint

`max_tokens` is a hard ceiling that the model cannot see. It does not sense the limit coming and
wrap up. Generation is cut mid-stream, wherever it happened to be.

This is the number one cause of "the model returned invalid JSON". The JSON was never invalid.
It was truncated at character 4096, and the tell was sitting in `stop_reason` the whole time, one
field away from the parse that blew up. Run the next cell to watch it happen.

Do not lowball the ceiling either: default to around 16000 non-streaming and 64000 streaming. The
small numbers in tutorials exist to keep demos cheap, this notebook included.


In [ ]:
# ── Truncation, and the field that would have warned you ─────
truncated = type("Response", (), {})()
truncated.stop_reason = "max_tokens"
truncated.text        = '{"summary": "timeouts dominate the log", "coun'

try:
    json.loads(truncated.text)          # the classic "the model returned invalid JSON"
except json.JSONDecodeError as exc:
    print("JSONDecodeError:", exc)

print("stop_reason was:", truncated.stop_reason, "-> the content was never complete")

# The model-aware alternative to a bigger ceiling is a task budget (beta): the server
# injects a countdown the model can see, so it paces itself and lands the plane.
budgeted = {"betas": ["task-budgets-2026-03-13"],
            "output_config": {"effort": "high",
                              "task_budget": {"type": "tokens", "total": 64000}}}  # min 20000


### Lesson: `stop_reason`, or why did this call end

Every response answers that question, and there is more than one honest answer. Like a phone call:
the other person finished their sentence (`end_turn`), their battery died mid-word (`max_tokens`),
they hit a word you had agreed would end the call (`stop_sequence`), they need to go look something
up (`tool_use`), or they declined to answer (`refusal`).

Nobody transcribes a call without first checking which of those happened. Treat a dead battery like
a polite goodbye and you will parse half a thought as though it were the whole one. Branch on
`stop_reason` before you touch `content`.

One detail worth remembering: `stop_details` is populated only for `refusal`, and is `None`
everywhere else, so guard before reading `.category`.


In [ ]:
# ── The dispatch every production caller needs ───────────────
STOP_REASONS = {
    "end_turn":      "finished naturally, safe to parse",
    "max_tokens":    "TRUNCATED, output is incomplete, do not parse as JSON",
    "stop_sequence": "hit one of your stop_sequences (the sequence is not in the text)",
    "tool_use":      "wants a tool, execute it and send tool_result back (module 2)",
    "pause_turn":    "long-running server tool paused, resend to resume",
    "refusal":       "safety decline, stop_details.category names the classifier",
}
for reason, meaning in STOP_REASONS.items():
    print(f"{reason:<14} {meaning}")

# ── stop_sequences: cut generation on a marker ───────────────
resp = call(
    model=MODEL,
    max_tokens=DEMO_MAX_TOKENS,
    stop_sequences=["\n\nHuman:", "</answer>"],   # the matched sequence is NOT in the returned text
    messages=[{"role": "user", "content": "Write one line, then </answer>."}],
)
if resp is not None:
    print(repr(text_of(resp)), "| stop_reason:", resp.stop_reason, "| matched:", resp.stop_sequence)


### 🏋️ Exercises 1.2


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 (warm-up) — When is the content complete?
# ════════════════════════════════════════════════════════
# Of the stop_reason values below, which ones mean the content is finished
# and safe to parse? Put those in SAFE. Nothing to implement.
#
#   end_turn   max_tokens   stop_sequence   refusal   pause_turn

SAFE = set()

check("safe to parse", SAFE, {"end_turn", "stop_sequence"})

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# SAFE = {"end_turn", "stop_sequence"}
#   max_tokens was cut mid-stream, whatever you got is a fragment.
#   pause_turn means the turn is not over, you resend to resume it.
#   refusal means there is no answer to parse in the first place.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 (fill in the gaps) — Branch before you parse
# ════════════════════════════════════════════════════════
# Fill the two stop_reason values so handle() refuses to hand back content
# that is incomplete or absent.

class Resp:
    def __init__(self, stop_reason, text):
        self.stop_reason, self.text = stop_reason, text

def handle(response):
    if response.stop_reason == ...:      # (1) the model declined to answer
        raise RuntimeError("refused")
    if response.stop_reason == ...:      # (2) generation was cut mid-stream
        raise ValueError("truncated, raise max_tokens or use a task budget")
    return response.text

def outcome(r):
    try:
        return handle(r)
    except Exception as exc:
        return type(exc).__name__

check("natural end", outcome(Resp("end_turn",   "all good")),      "all good")
check("truncated",   outcome(Resp("max_tokens", '{\"coun')),        "ValueError")
check("refusal",     outcome(Resp("refusal",    "")),              "RuntimeError")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def handle(response):
#     if response.stop_reason == "refusal":
#         raise RuntimeError("refused")
#     if response.stop_reason == "max_tokens":
#         raise ValueError("truncated, raise max_tokens or use a task budget")
#     return response.text


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Reproduce how the server decides stop_reason
# ════════════════════════════════════════════════════════
# simulate_stop(text, stop_sequences, max_chars) -> dict with keys
# "text", "stop_reason", "stop_sequence". Rules:
#   • the earliest matching stop sequence cuts the text just before it, and
#     the sequence itself is never returned    -> "stop_sequence"
#   • otherwise, text longer than max_chars is cut to max_chars -> "max_tokens"
#   • otherwise the text is returned whole     -> "end_turn"
#   • stop_sequence is the matched string, or None

def simulate_stop(text, stop_sequences, max_chars):
    pass  # to complete

body = "Answer: 42.\n\nHuman: and then?"
check("stop hit",  simulate_stop(body, ["\n\nHuman:"], 200),
      {"text": "Answer: 42.", "stop_reason": "stop_sequence", "stop_sequence": "\n\nHuman:"})
check("truncated", simulate_stop("a" * 50, [], 8),
      {"text": "a" * 8, "stop_reason": "max_tokens", "stop_sequence": None})
check("clean end", simulate_stop("short and clean", [], 200),
      {"text": "short and clean", "stop_reason": "end_turn", "stop_sequence": None})

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def simulate_stop(text, stop_sequences, max_chars):
#     hits = [(text.find(s), s) for s in stop_sequences if s in text]
#     if hits:
#         pos, seq = min(hits)
#         return {"text": text[:pos], "stop_reason": "stop_sequence", "stop_sequence": seq}
#     if len(text) > max_chars:
#         return {"text": text[:max_chars], "stop_reason": "max_tokens", "stop_sequence": None}
#     return {"text": text, "stop_reason": "end_turn", "stop_sequence": None}


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Implement the decoding step the API used to expose, using softmax() above.
# nucleus_sample(logits, temperature, top_p, rng) -> (chosen, kept, probs)
#   1. temperature, then softmax
#   2. sort indices by descending probability
#   3. keep the smallest prefix whose cumulative probability reaches top_p
#      (always keep at least one)
#   4. renormalise the kept probabilities to sum to 1
#   5. draw with inverse-CDF sampling using rng.random()
# `chosen` indexes into the ORIGINAL logits; `kept` is in descending-probability order.

def nucleus_sample(logits, temperature, top_p, rng):
    pass  # to complete

got = nucleus_sample([3.2, 2.9, 1.1, 0.4, -0.8], 1.0, 0.8, random.Random(0))
check("kept indices", lambda: got[1], [0, 1])
check("renormalised", lambda: [round(p, 4) for p in got[2]], [0.5744, 0.4256])
check("drawn index",  lambda: got[0], 1)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def nucleus_sample(logits, temperature, top_p, rng):
#     probs = softmax(logits, temperature)
#     order = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
#     kept, cumulative = [], 0.0
#     for i in order:
#         kept.append(i)
#         cumulative += probs[i]
#         if cumulative >= top_p:
#             break
#     total = sum(probs[i] for i in kept)
#     kept_probs = [probs[i] / total for i in kept]
#     r, acc = rng.random(), 0.0
#     for i, p in zip(kept, kept_probs):
#         acc += p
#         if r <= acc:
#             return i, kept, kept_probs
#     return kept[-1], kept, kept_probs


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# A gateway receives legacy kwargs and must make them valid for the target
# model instead of returning a 400.
# normalise_request(model, **kwargs) -> (clean_kwargs, warnings), applying in order:
#   1. drop temperature / top_p / top_k on SAMPLING_REMOVED models
#                                       -> "<param> removed on <model>"
#   2. thinking {"type": "enabled", "budget_tokens": N} on those models becomes
#      {"type": "adaptive"}             -> "budget_tokens removed on <model>"
#   3. effort "xhigh" on a model outside XHIGH_MODELS downgrades to "high"
#                                       -> "xhigh unavailable on <model>"
#   4. thinking {"type": "disabled"} together with effort in {"xhigh", "max"} is a
#      400 on Opus 5, so drop the thinking key
#                                       -> "disabled thinking rejected at <effort>"
# Do not mutate the caller's dicts.

SAMPLING_REMOVED = {"claude-opus-5", "claude-opus-4-8", "claude-opus-4-7",
                    "claude-sonnet-5", "claude-fable-5"}
XHIGH_MODELS     = {"claude-opus-5", "claude-opus-4-8", "claude-opus-4-7",
                    "claude-sonnet-5", "claude-fable-5"}

def normalise_request(model, **kwargs):
    pass  # to complete

got = normalise_request(
    "claude-opus-5", temperature=0.0, top_p=0.9,
    thinking={"type": "enabled", "budget_tokens": 4096},
    output_config={"effort": "xhigh"}, max_tokens=16000,
)
check("clean kwargs", lambda: got[0], {"thinking": {"type": "adaptive"},
                              "output_config": {"effort": "xhigh"},
                              "max_tokens": 16000})
check("warnings", lambda: got[1], ["temperature removed on claude-opus-5",
                          "top_p removed on claude-opus-5",
                          "budget_tokens removed on claude-opus-5"])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def normalise_request(model, **kwargs):
#     clean, warnings = dict(kwargs), []
#     for param in ("temperature", "top_p", "top_k"):
#         if param in clean and model in SAMPLING_REMOVED:
#             clean.pop(param)
#             warnings.append(f"{param} removed on {model}")
#     thinking = clean.get("thinking")
#     if isinstance(thinking, dict):
#         thinking = dict(thinking)
#         if thinking.get("type") == "enabled" and model in SAMPLING_REMOVED:
#             thinking = {"type": "adaptive"}
#             warnings.append(f"budget_tokens removed on {model}")
#         clean["thinking"] = thinking
#     oc = clean.get("output_config")
#     if isinstance(oc, dict):
#         oc = dict(oc)
#         if oc.get("effort") == "xhigh" and model not in XHIGH_MODELS:
#             oc["effort"] = "high"
#             warnings.append(f"xhigh unavailable on {model}")
#         clean["output_config"] = oc
#     effort = clean.get("output_config", {}).get("effort")
#     if clean.get("thinking", {}).get("type") == "disabled" and effort in {"xhigh", "max"}:
#         clean.pop("thinking")
#         warnings.append(f"disabled thinking rejected at {effort}")
#     return clean, warnings


---
## 1.3 Tokens and the context window

Tokens are the unit of cost, of latency, and of the ceiling on how much you can send. Every
budgeting decision an agent makes is arithmetic on token counts, which makes this section mostly
about two things: getting those counts from the right place, and reading the `usage` object
without being misled by the one field that misleads everybody. You finish able to price a run and
to trim a conversation that has outgrown its window without corrupting it.

In [ ]:
# ── Count before you send ────────────────────────────────────
document = "Retrieval augmented generation " * 40

if LIVE:
    counted = CLIENT.messages.count_tokens(
        model=MODEL,
        system="You are a summariser.",
        messages=[{"role": "user", "content": document}],
    )
    print("input_tokens:", counted.input_tokens)
else:
    print("[offline] POST /v1/messages/count_tokens, same body shape as messages.create")


def rough_tokens(text: str) -> int:
    "Offline stand-in ONLY. Never ship this: it is wrong for code, JSON and non-English."
    return max(1, len(text) // 4)


print("rough estimate:", rough_tokens(document), "tokens (indicative, not a budget)")


### Lesson: A token is a slice, and every tokenizer slices differently

Tokens are the unit of cost, of latency, and of the context limit. Every budgeting decision an
agent makes is arithmetic on token counts, so wrong counts mean wrong budgets everywhere
downstream.

Think of a loaf of bread. `tiktoken` cuts it the way OpenAI's models like it. Claude's tokenizer
cuts the same loaf into different pieces, usually more of them. Measuring Claude's context window
in `tiktoken` slices is like checking whether a cake fits its box by counting some other bakery's
slices: the number looks precise, it is quietly wrong, and you find out the day the box is full.

Concretely, `tiktoken` undercounts Claude tokens by roughly 15 to 20% on ordinary prose, and by far
more on code, JSON and non-English text. Use `client.messages.count_tokens()` instead. It takes the
same body as `messages.create`, and counts are model-specific, so pass the model you will call.


In [ ]:
# ── Context and output limits per model ──────────────────────
LIMITS = {
    "claude-opus-5":     {"context": 1_000_000, "max_output": 128_000},
    "claude-sonnet-5":   {"context": 1_000_000, "max_output": 128_000},
    "claude-haiku-4-5":  {"context":   200_000, "max_output":   8_192},
}
for name, lim in LIMITS.items():
    print(f"{name:<20} context={lim['context']:>9,}  max_output={lim['max_output']:>7,}")

# The context window covers the WHOLE request: system + tools + every message, plus the
# reply being generated. Budget for max_tokens too, not only for what you send.


### Lesson: `usage.input_tokens` is not your prompt size

Every response carries a `usage` object, and it is the only honest signal about what a run cost.
It also has one field that misleads almost everybody on first read.

`input_tokens` is the *uncached remainder*, not the prompt. The real prompt is
`input_tokens + cache_creation_input_tokens + cache_read_input_tokens`. Reading only the first
number is like reading a grocery receipt that lists what you paid for today and silently omits
everything you took from the pantry. Teams report "our agent only sends 4k tokens" while actually
sending 200k, 196k of it served from cache.

The next cell prints both numbers side by side.


In [ ]:
# ── The same turn, read two ways ─────────────────────────────
usage = {
    "input_tokens": 1_200,                  # billed at full price
    "cache_creation_input_tokens": 8_000,   # written to cache this turn (~1.25x price)
    "cache_read_input_tokens": 190_000,     # served from cache (~0.1x price)
    "output_tokens": 640,
}
prompt_size = (usage["input_tokens"]
               + usage["cache_creation_input_tokens"]
               + usage["cache_read_input_tokens"])
print(f"what input_tokens showed : {usage['input_tokens']:,}")
print(f"what you actually sent   : {prompt_size:,}")

# ── Pricing, $ per 1M tokens ─────────────────────────────────
PRICES = {
    "claude-opus-5":    {"input": 5.00, "output": 25.00},
    "claude-sonnet-5":  {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5": {"input": 1.00, "output":  5.00},
}
CACHE_WRITE_MULT = {"5m": 1.25, "1h": 2.00}   # premium paid once, on write
CACHE_READ_MULT  = 0.10                       # cache reads cost ~10% of base input


### Lesson: Trimming history without breaking the transcript

Agent loops resend the whole transcript every turn, so spend grows quadratically with turn count.
Sooner or later you have to drop something, and dropping carelessly is its own failure mode.

Picture editing a play down to its final scenes. Cut without looking and an actor is left on stage
reacting to "the wrench I asked you to check", a prop that only existed in a scene you removed. The
API catches the structural version of this immediately: `messages[0]` must be a user turn, and a
`tool_use` block must keep the `tool_result` that answers it.

Three ways to stay inside the window, in increasing sophistication: a sliding window (cheap and
forgetful), context editing (`clear_tool_uses_20250919`, which clears old tool results), and
compaction (`compact_20260112`, which summarises earlier context server-side). Module 4 builds all
three by hand.


In [ ]:
# ── A sliding window that respects the structural rules ──────
def sliding_window(messages: list[dict], keep_last: int) -> list[dict]:
    "Keep the tail, then repair the head so messages[0] is a user turn."
    kept = messages[-keep_last:]
    while kept and kept[0]["role"] != "user":
        kept = kept[1:]
    return kept


convo = [{"role": r, "content": f"turn {i}"}
         for i, r in enumerate(["user", "assistant"] * 5)]
print("raw tail  :", [m["role"] for m in convo[-5:]])
print("repaired  :", [m["role"] for m in sliding_window(convo, keep_last=5)])

# ── Server-side alternatives (module 4 goes deeper) ──────────
context_managed = {"betas": ["context-management-2025-06-27"],
                   "context_management": {"edits": [{"type": "clear_tool_uses_20250919"}]}}
compacted       = {"betas": ["compact-2026-01-12"],
                   "context_management": {"edits": [{"type": "compact_20260112"}]}}


### 🏋️ Exercises 1.3


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 (warm-up) — How big was that prompt really?
# ════════════════════════════════════════════════════════
# One expression, using the fields of `turn` below. Nothing else to implement.

turn = {"input_tokens": 3_100,
        "cache_creation_input_tokens": 12_000,
        "cache_read_input_tokens": 145_000,
        "output_tokens": 900}

PROMPT_TOKENS = 0

check("real prompt size", PROMPT_TOKENS, 160_100)

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# PROMPT_TOKENS = (turn["input_tokens"]
#                  + turn["cache_creation_input_tokens"]
#                  + turn["cache_read_input_tokens"])
#   output_tokens is not part of the prompt: it is what came back.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 (fill in the gaps) — What did this turn cost?
# ════════════════════════════════════════════════════════
# The formula is written. Replace the two placeholder multipliers with the
# right ones from the lesson (CACHE_WRITE_MULT, CACHE_READ_MULT).

def turn_cost(usage, model, ttl="5m"):
    inp, out = PRICES[model]["input"], PRICES[model]["output"]
    dollars = (usage["input_tokens"]               * inp
               + usage["cache_creation_input_tokens"] * inp * 1.0     # (1) write premium
               + usage["cache_read_input_tokens"]     * inp * 1.0     # (2) read discount
               + usage["output_tokens"]                * out)
    return round(dollars / 1_000_000, 6)

check("5m ttl", turn_cost(usage, "claude-opus-5"), 0.167)
check("1h ttl", turn_cost(usage, "claude-opus-5", ttl="1h"), 0.197)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def turn_cost(usage, model, ttl="5m"):
#     inp, out = PRICES[model]["input"], PRICES[model]["output"]
#     dollars = (usage["input_tokens"]               * inp
#                + usage["cache_creation_input_tokens"] * inp * CACHE_WRITE_MULT[ttl]
#                + usage["cache_read_input_tokens"]     * inp * CACHE_READ_MULT
#                + usage["output_tokens"]                * out)
#     return round(dollars / 1_000_000, 6)


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Fit a transcript into a token budget
# ════════════════════════════════════════════════════════
# fit_history(messages, budget) -> list[dict]. Each message is
# {"role": ..., "tokens": int}. Rules:
#   • walk from the END backwards, keeping messages while the running total
#     stays within budget
#   • the last message is ALWAYS kept, even if it alone busts the budget
#   • the result must start on a "user" turn, so drop leading non-user ones
#   • keep the original order

history = [
    {"role": "user",      "tokens": 500},
    {"role": "assistant", "tokens": 900},
    {"role": "user",      "tokens": 300},
    {"role": "assistant", "tokens": 700},
    {"role": "user",      "tokens": 200},
]

def fit_history(messages, budget):
    pass  # to complete

check("budget 1300", lambda: [(m["role"], m["tokens"]) for m in fit_history(history, 1300)],
      [("user", 300), ("assistant", 700), ("user", 200)])
check("budget 100",  lambda: [(m["role"], m["tokens"]) for m in fit_history(history, 100)],
      [("user", 200)])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def fit_history(messages, budget):
#     kept, total = [], 0
#     for m in reversed(messages):
#         if kept and total + m["tokens"] > budget:
#             break
#         kept.append(m)
#         total += m["tokens"]
#     kept.reverse()
#     while len(kept) > 1 and kept[0]["role"] != "user":
#         kept.pop(0)
#     return kept


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Token-aware chunking with overlap, the sizing step of module 4's RAG pipeline.
# chunk_by_tokens(text, count_fn, max_tokens, overlap_tokens) -> list[str].
# `count_fn(str) -> int` is INJECTED so production can pass
# client.messages.count_tokens while tests pass a deterministic stub.
#   • split on whitespace, greedily pack words while count_fn(chunk) <= max_tokens
#   • each new chunk restarts with as many trailing words of the previous chunk
#     as fit within overlap_tokens
#   • a single word over max_tokens still becomes its own chunk

def chunk_by_tokens(text, count_fn, max_tokens, overlap_tokens):
    pass  # to complete

by_word = lambda s: len(s.split())        # stub counter: 1 token per word
check("chunks", chunk_by_tokens(" ".join(f"w{i}" for i in range(10)), by_word, 4, 1),
      ["w0 w1 w2 w3", "w3 w4 w5 w6", "w6 w7 w8 w9"])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def chunk_by_tokens(text, count_fn, max_tokens, overlap_tokens):
#     assert overlap_tokens < max_tokens, "overlap must be smaller than the chunk"
#     words, chunks, current = text.split(), [], []
#     for word in words:
#         candidate = current + [word]
#         if current and count_fn(" ".join(candidate)) > max_tokens:
#             chunks.append(" ".join(current))
#             tail = []
#             for w in reversed(current):
#                 if count_fn(" ".join([w] + tail)) > overlap_tokens:
#                     break
#                 tail.insert(0, w)
#             current = tail + [word]
#         else:
#             current = candidate
#     if current:
#         chunks.append(" ".join(current))
#     return chunks


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# The full ledger for a whole run, not one turn.
# run_cost(usages, model) -> dict with keys "input", "cache_write",
# "cache_read", "output", "total" (dollars, rounded to 6 decimals) and
# "prompt_tokens" (integer sum of the three input fields over every turn).
# Each usage may carry "ttl", defaulting to "5m".

usages = [
    {"input_tokens": 12_000, "cache_creation_input_tokens": 30_000,
     "cache_read_input_tokens": 0, "output_tokens": 800},
    {"input_tokens": 400, "cache_creation_input_tokens": 0,
     "cache_read_input_tokens": 30_000, "output_tokens": 1_200},
    {"input_tokens": 350, "cache_creation_input_tokens": 5_000,
     "cache_read_input_tokens": 30_000, "output_tokens": 900, "ttl": "1h"},
]

def run_cost(usages, model):
    pass  # to complete

check("ledger", run_cost(usages, "claude-opus-5"),
      {"input": 0.06375, "cache_write": 0.2375, "cache_read": 0.03,
       "output": 0.0725, "total": 0.40375, "prompt_tokens": 107_750})

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def run_cost(usages, model):
#     p = PRICES[model]
#     acc = {"input": 0.0, "cache_write": 0.0, "cache_read": 0.0, "output": 0.0}
#     prompt_tokens = 0
#     for u in usages:
#         mult = CACHE_WRITE_MULT[u.get("ttl", "5m")]
#         acc["input"]       += u["input_tokens"] * p["input"]
#         acc["cache_write"] += u["cache_creation_input_tokens"] * p["input"] * mult
#         acc["cache_read"]  += u["cache_read_input_tokens"] * p["input"] * CACHE_READ_MULT
#         acc["output"]      += u["output_tokens"] * p["output"]
#         prompt_tokens += (u["input_tokens"] + u["cache_creation_input_tokens"]
#                           + u["cache_read_input_tokens"])
#     out = {k: round(v / 1_000_000, 6) for k, v in acc.items()}
#     out["total"] = round(sum(out.values()), 6)
#     out["prompt_tokens"] = prompt_tokens
#     return out


---
## 1.4 Streaming

A non-streaming call gives you nothing until the final token exists. For anything a person is
waiting on that is unacceptable, and for long generations it is not even permitted. Streaming
turns the request into a sequence of events, which is both more useful and more fiddly: several
blocks arrive interleaved, and part of what arrives is not safe to act on yet. Telling those apart
is the skill this section builds.

In [ ]:
# ── Streaming, the convenience path ──────────────────────────
if LIVE:
    with CLIENT.messages.stream(
        model=MODEL,
        max_tokens=DEMO_MAX_TOKENS,
        messages=[{"role": "user", "content": "Name three agent failure modes."}],
    ) as stream:
        for chunk in stream.text_stream:          # text deltas only, already filtered
            print(chunk, end="", flush=True)
        final = stream.get_final_message()        # the full Message once the stream ends
    print("\n\nstop_reason:", final.stop_reason, "| output:", final.usage.output_tokens)
else:
    print("[offline] client.messages.stream(...) -> text_stream / get_final_message()")


### Lesson: Why stream at all

A non-streaming call is a letter that reaches your mailbox only once it is finished and sealed:
silence, then everything at once. For a 2000-token answer that silence runs 20 to 40 seconds.
Streaming is watching through the window while it gets written, word by word.

That is the obvious reason. The less obvious one is that streaming is a hard requirement, not a
nicety. Idle proxies and load balancers cheerfully kill a connection that goes quiet for 30
seconds, and the SDK itself refuses a non-streaming request whose `max_tokens` it estimates will
run past roughly ten minutes, raising a `ValueError` before any HTTP call happens. Since current
models allow up to 128k output tokens, any generous ceiling forces you onto `.stream()`.


In [ ]:
# ── The six event types, in the order they fire ──────────────
EVENT_SEQUENCE = [
    ("message_start",        "message metadata + input usage, fires once"),
    ("content_block_start",  "a block begins; content_block.type says which kind"),
    ("content_block_delta",  "incremental payload; delta.type varies by block"),
    ("content_block_stop",   "the block is complete, and only now is its JSON parseable"),
    ("message_delta",        "top-level updates: stop_reason and the FINAL output_tokens"),
    ("message_stop",         "end of stream"),
]
for name, meaning in EVENT_SEQUENCE:
    print(f"{name:<20} {meaning}")

# delta.type by block type:
#   text      -> text_delta        (.text)
#   thinking  -> thinking_delta    (.thinking)      empty unless display="summarized"
#   tool_use  -> input_json_delta  (.partial_json)  a JSON FRAGMENT, not JSON


def delta_payload(delta: dict) -> str:
    "Every delta type carries its payload under a different key. Normalise them."
    return delta.get("text") or delta.get("thinking") or delta.get("partial_json") or ""


### Lesson: One response, several blocks, addressed by index

Here is the part people miss. The writer at that window is filling several notebooks at once: a
scratch pad for reasoning, a clean sheet for the reply, an order form for a tool call. Each is
tagged with an `index`, and what arrives on the wire reads like "notebook 2, new page", "notebook
2, new word", "notebook 2, page closed".

So a single response interleaves `thinking`, `text` and `tool_use` blocks, each with its own
start / delta / stop triplet. Code that concatenates every delta into one string glues the model's
private reasoning onto the front of its answer. Keep one accumulator per index. The next cell shows
both versions side by side.


In [ ]:
# ── A recorded stream carrying reasoning AND an answer ───────
MIXED = [
    {"type": "content_block_start", "index": 0, "content_block": {"type": "thinking"}},
    {"type": "content_block_delta", "index": 0,
     "delta": {"type": "thinking_delta", "thinking": "They probably want the short version. "}},
    {"type": "content_block_stop", "index": 0},
    {"type": "content_block_start", "index": 1, "content_block": {"type": "text"}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "text_delta", "text": "Retry storms happen when every client "}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "text_delta", "text": "retries at the same instant."}},
    {"type": "content_block_stop", "index": 1},
]

naive = "".join(delta_payload(e["delta"]) for e in MIXED if e["type"] == "content_block_delta")
print("naive :", naive)          # reasoning glued to the front of the answer

per_index: dict[int, list] = {}
for e in MIXED:
    if e["type"] == "content_block_delta":
        per_index.setdefault(e["index"], []).append(delta_payload(e["delta"]))
print("index0:", "".join(per_index[0]))
print("index1:", "".join(per_index[1]))


### Lesson: `partial_json` is a fragment, not JSON

Tool arguments arrive as `input_json_delta` fragments: `{"ci`, then `ty": "Par`, then `is"}`.
Reading them mid-flight is like reading a fax while it is still sliding out of the machine. The
line forming right now says "do not", and three seconds later it says "do not forget".

Calling `json.loads` on the buffer throws on every fragment but the last, which is how people end
up wrapping it in a bare `except: pass` and then wondering why a tool occasionally fires with half
its arguments. Accumulate the fragments, parse only at `content_block_stop`.

One more, easy to miss: `message_start` reports `output_tokens: 1`. The real count arrives in
`message_delta` at the end, so reading it from the start event under-reports every run.


In [ ]:
# ── Parsing too early, one fragment at a time ────────────────
FRAGMENTS = ['{"ci', 'ty": "Par', 'is", "unit"', ': "c"}']

buf = ""
for frag in FRAGMENTS:
    buf += frag
    try:
        json.loads(buf)
        print(f"{buf!r:<34} parses")
    except json.JSONDecodeError:
        print(f"{buf!r:<34} not yet")

# Only the final buffer is a real object. That is what content_block_stop tells you.

# ── An interrupted stream is NOT a turn ──────────────────────
# On APIConnectionError mid-stream you hold partial content. Retry the turn; never
# append the fragment to `messages` as an assistant turn, or you teach the model
# that truncated answers are normal.


### 🏋️ Exercises 1.4


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 (warm-up) — Where does the real token count live?
# ════════════════════════════════════════════════════════
# You want the FINAL output_tokens for a streamed response. Which event
# carries it? Set ANSWER to the event type. Nothing to implement.

ANSWER = ""     # message_start | content_block_stop | message_delta | message_stop

check("final output_tokens", ANSWER, "message_delta")

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# ANSWER = "message_delta"
#   message_start reports output_tokens: 1, a placeholder, because generation has
#   barely begun. Reading the count there under-reports every single run.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 (fill in the gaps) — Collect only the answer text
# ════════════════════════════════════════════════════════
# Fill the two placeholders so collect_text returns the reply and leaves the
# model's reasoning out of it. MIXED comes from the lesson above.

def collect_text(events):
    buf = []
    for e in events:
        if e["type"] == "content_block_delta" and e["delta"]["type"] == "?":   # (1)
            buf.append(e["delta"]["?"])                                        # (2)
    return "".join(buf)

check("answer only", collect_text(MIXED),
      "Retry storms happen when every client retries at the same instant.")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def collect_text(events):
#     buf = []
#     for e in events:
#         if e["type"] == "content_block_delta" and e["delta"]["type"] == "text_delta":
#             buf.append(e["delta"]["text"])
#     return "".join(buf)


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Rebuild the content blocks from raw events
# ════════════════════════════════════════════════════════
# RECORDED is a faithful capture of the SSE events for one response.
# accumulate_content(events) -> the `content` list a non-streaming call
# would have returned:
#   • one accumulator per index, blocks returned in index order
#   • a text block ends up as {"type": "text", "text": <joined deltas>}
#   • a tool_use block keeps its id and name, and its "input" is the joined
#     fragments PARSED, which is only safe at content_block_stop
# `delta_payload` from the lesson gives you the payload of any delta.

RECORDED = [
    {"type": "message_start", "message": {
        "id": "msg_017x", "model": "claude-opus-5",
        "usage": {"input_tokens": 1450, "output_tokens": 1}}},
    {"type": "content_block_start", "index": 0,
     "content_block": {"type": "text", "text": ""}},
    {"type": "content_block_delta", "index": 0,
     "delta": {"type": "text_delta", "text": "Let me check "}},
    {"type": "content_block_delta", "index": 0,
     "delta": {"type": "text_delta", "text": "the weather."}},
    {"type": "content_block_stop", "index": 0},
    {"type": "content_block_start", "index": 1,
     "content_block": {"type": "tool_use", "id": "toolu_09", "name": "get_weather", "input": {}}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "input_json_delta", "partial_json": '{"ci'}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "input_json_delta", "partial_json": 'ty": "Par'}},
    {"type": "content_block_delta", "index": 1,
     "delta": {"type": "input_json_delta", "partial_json": 'is", "unit": "c"}'}},
    {"type": "content_block_stop", "index": 1},
    {"type": "message_delta", "delta": {"stop_reason": "tool_use"},
     "usage": {"output_tokens": 57}},
    {"type": "message_stop"},
]

def accumulate_content(events):
    pass  # to complete

check("rebuilt content", accumulate_content(RECORDED), [
    {"type": "text", "text": "Let me check the weather."},
    {"type": "tool_use", "id": "toolu_09", "name": "get_weather",
     "input": {"city": "Paris", "unit": "c"}},
])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def accumulate_content(events):
#     acc = {}
#     for e in events:
#         if e["type"] == "content_block_start":
#             acc[e["index"]] = {"block": dict(e["content_block"]), "buf": []}
#         elif e["type"] == "content_block_delta":
#             acc[e["index"]]["buf"].append(delta_payload(e["delta"]))
#         elif e["type"] == "content_block_stop":
#             entry = acc[e["index"]]
#             raw = "".join(entry["buf"])
#             if entry["block"]["type"] == "text":
#                 entry["block"]["text"] = raw
#             else:
#                 entry["block"]["input"] = json.loads(raw)   # safe ONLY here
#     return [acc[i]["block"] for i in sorted(acc)]


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Latency metrics for one streamed response. TIMED is (elapsed_seconds, event_type),
# t=0 being the moment the request was sent.
# stream_metrics(timed, output_tokens) -> dict with
#   "ttft"        : elapsed at the FIRST content_block_delta
#   "total"       : elapsed at message_stop
#   "generation"  : total - ttft
#   "tokens_per_s": output_tokens / generation, 0.0 when generation is 0
#   "stall_max"   : largest gap between consecutive content_block_delta events
# Round every float to 3 decimals.

TIMED = [
    (0.000, "message_start"),
    (0.412, "content_block_start"),
    (0.418, "content_block_delta"),
    (0.455, "content_block_delta"),
    (0.902, "content_block_delta"),   # a stall, upstream hiccup
    (0.940, "content_block_delta"),
    (0.975, "content_block_stop"),
    (0.980, "message_delta"),
    (0.985, "message_stop"),
]

def stream_metrics(timed, output_tokens):
    pass  # to complete

check("metrics", stream_metrics(TIMED, 57),
      {"ttft": 0.418, "total": 0.985, "generation": 0.567,
       "tokens_per_s": 100.529, "stall_max": 0.447})

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def stream_metrics(timed, output_tokens):
#     deltas = [t for t, kind in timed if kind == "content_block_delta"]
#     ttft = deltas[0] if deltas else 0.0
#     total = next(t for t, kind in timed if kind == "message_stop")
#     generation = total - ttft
#     gaps = [b - a for a, b in zip(deltas, deltas[1:])]
#     return {"ttft": round(ttft, 3),
#             "total": round(total, 3),
#             "generation": round(generation, 3),
#             "tokens_per_s": round(output_tokens / generation, 3) if generation else 0.0,
#             "stall_max": round(max(gaps), 3) if gaps else 0.0}


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# The full reconstruction, not just the content list.
# accumulate(events) -> {"id", "model", "content", "stop_reason", "usage"},
# exactly what a non-streaming call would have returned. Reuse
# accumulate_content for the blocks; take id and model and input_tokens from
# message_start, stop_reason and output_tokens from message_delta.

def accumulate(events):
    pass  # to complete

check("full message", accumulate(RECORDED), {
    "id": "msg_017x",
    "model": "claude-opus-5",
    "content": [
        {"type": "text", "text": "Let me check the weather."},
        {"type": "tool_use", "id": "toolu_09", "name": "get_weather",
         "input": {"city": "Paris", "unit": "c"}},
    ],
    "stop_reason": "tool_use",
    "usage": {"input_tokens": 1450, "output_tokens": 57},
})

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def accumulate(events):
#     out = {"id": None, "model": None, "content": [], "stop_reason": None,
#            "usage": {"input_tokens": 0, "output_tokens": 0}}
#     for e in events:
#         if e["type"] == "message_start":
#             m = e["message"]
#             out["id"], out["model"] = m["id"], m["model"]
#             out["usage"]["input_tokens"] = m["usage"]["input_tokens"]
#         elif e["type"] == "message_delta":
#             out["stop_reason"] = e["delta"]["stop_reason"]
#             out["usage"]["output_tokens"] = e["usage"]["output_tokens"]
#     out["content"] = accumulate_content(events)
#     return out


---
## 1.5 Structured output

An agent is code, and code wants typed values rather than prose. Asking politely for JSON in the
system prompt works most of the time, and "most of the time" stops being a useful guarantee once a
single run makes twenty calls. This section covers the mechanism that makes the shape enforced
rather than merely requested, then what to do on the paths where that mechanism is not available
to you.

In [ ]:
# ── Ask for a typed object, get a typed object ───────────────
from pydantic import BaseModel, Field, ValidationError

class TriageResult(BaseModel):
    category: str = Field(description="billing | technical | account")
    severity: int = Field(ge=1, le=5)
    summary: str
    needs_human: bool

if LIVE:
    parsed = CLIENT.messages.parse(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user",
                   "content": "Ticket: card charged twice, customer furious, wants a callback."}],
        output_format=TriageResult,          # the SDK derives the schema and validates the reply
    )
    result = parsed.parsed_output            # a real TriageResult instance
    print(result.category, result.severity, result.needs_human)
else:
    print("[offline] client.messages.parse(..., output_format=TriageResult).parsed_output")


### Lesson: Constrained decoding beats prompt-and-pray

Asking for JSON in the system prompt is like asking a chatty friend to reply only in one-word
texts. Most of the time they will. Then one day they add "sure, here you go!" in front, because a
request is a suggestion and people may ignore suggestions.

Constrained decoding is not a politer version of that request, it is a different mechanism. Think
of a form with pre-printed boxes, one character per box, where the shape is enforced by the
printing press rather than by whoever fills it in. The model is not being well mannered about the
format. It is structurally unable to emit a token that breaks the schema.

That matters more than it sounds. "Works 98% of the time" in an agent making 20 calls per run means
roughly a third of runs contain a parse failure.


In [ ]:
# ── The raw wire form the SDK builds for you ─────────────────
output_format = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "category":    {"type": "string", "enum": ["billing", "technical", "account"]},
            "severity":    {"type": "integer"},
            "summary":     {"type": "string"},
            "needs_human": {"type": "boolean"},
        },
        "required": ["category", "severity", "summary", "needs_human"],   # ALL of them
        "additionalProperties": False,                                    # mandatory
    },
}
print(json.dumps(output_format, indent=1)[:280])

# ── Strict tool arguments: same guarantee, tool side ─────────
strict_tool = {
    "name": "create_ticket",
    "description": "File a support ticket",
    "strict": True,                            # top-level, NOT inside tool_choice
    "input_schema": {
        "type": "object",
        "properties": {"title": {"type": "string"}, "priority": {"type": "integer"}},
        "required": ["title", "priority"],
        "additionalProperties": False,
    },
}

# The 2024 trick of prefilling the assistant turn with "{" to force JSON is REMOVED:
#   messages=[..., {"role": "assistant", "content": "{"}]     -> 400 on every current model


### Lesson: A schema does not save you from `max_tokens`

Two traps sit right behind constrained decoding, and both look like the schema failing when it did
not.

The first: a strict schema needs `"additionalProperties": false` **and** a complete `required`
list, on every object node. Leave either out and "strict" quietly means "mostly". The second: the
ceiling from 1.2 still applies. Constrained output that runs past `max_tokens` gives you
schema-valid-until-truncated JSON, which parses as a syntax error rather than a schema error, so
the `stop_reason` check stays mandatory even here.

Worth filing away too: `output_config.format` is incompatible with citations, and returns a 400 if
you send both.


In [ ]:
# ── Schema-valid right up to the moment it was cut ───────────
constrained_but_truncated = type("Response", (), {})()
constrained_but_truncated.stop_reason = "max_tokens"
constrained_but_truncated.text = '{"category": "billing", "severity": 4, "summ'

try:
    json.loads(constrained_but_truncated.text)
except json.JSONDecodeError as exc:
    print("JSONDecodeError:", exc.msg, "| stop_reason:", constrained_but_truncated.stop_reason)

# The schema was never violated. The response just stopped existing halfway through.


### Lesson: Defensive parsing for everything else

Constrained decoding is the seatbelt in a brand-new car, and you will not only ever drive
brand-new cars. Older models, third-party gateways that strip `output_config`, and other agents in
a pipeline all hand you "probably JSON" with no seatbelt fitted.

So you still need a repair-and-validate stage, plus one habit that pays for itself: when
validation fails, resend with the validation error in the message. Handing a student's paper back
marked only "wrong, try again" gets you the same mistake twice. Circle the line and say why, and it
gets fixed. A model corrects an error it can read, not one it has to guess at. Cap the attempts
though, because an uncapped repair loop is the classic runaway-cost incident.

One more rule, small and absolute: never string-match a serialised tool input. Escaping differs
between models. Parse it.


In [ ]:
# ── A pragmatic parser for untrusted "probably JSON" ─────────
def loads_or_none(text: str):
    "Strip a code fence, take the outermost braces, parse. Never raises."
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(cleaned[start:end + 1])
    except json.JSONDecodeError:
        return None


for raw in ['{"category": "billing", "severity": 4}',
            'Sure! Here you go:\n```json\n{"category": "billing"}\n```',
            'The answer is {"category": "technical"}, hope that helps!',
            'no json at all']:
    print(f"{raw[:38]!r:<42} -> {loads_or_none(raw)}")


# ── Turn a validation error into something actionable ────────
def validation_feedback(exc: ValidationError) -> str:
    lines = [f"- {'.'.join(str(p) for p in e['loc'])}: {e['msg']}" for e in exc.errors()]
    return "Your JSON failed validation:\n" + "\n".join(lines) + "\nReturn corrected JSON only."


try:
    TriageResult.model_validate({"category": "billing", "severity": 9, "summary": "x"})
except ValidationError as exc:
    print()
    print(validation_feedback(exc))

# ── Never regex a serialised tool input ──────────────────────
serialised = '{"path": "\\/var\\/log\\/app.log"}'          # legal JSON escaping of "/"
print("substring match:", '"/var/log/app.log"' in serialised)   # False, brittle
print("parsed         :", json.loads(serialised)["path"])       # /var/log/app.log


### 🏋️ Exercises 1.5


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 (warm-up) — What makes a schema strict?
# ════════════════════════════════════════════════════════
# The API accepts a schema as strict only when every object node declares two
# particular keys. Put their names in MISSING. Nothing to implement.

loose = {
    "type": "object",
    "properties": {"title": {"type": "string"}, "priority": {"type": "integer"}},
}

MISSING = set()

check("keys a strict object node needs", MISSING, {"required", "additionalProperties"})

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# MISSING = {"required", "additionalProperties"}
#   "required" must list EVERY property, not just the ones you consider mandatory.
#   "additionalProperties": false is what forbids invented extra keys.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 (fill in the gaps) — Make this schema strict
# ════════════════════════════════════════════════════════
# Replace the two placeholder values so `schema` passes is_strict().

def is_strict(node):
    "The check the API effectively applies to an object node."
    return (node.get("additionalProperties") is False
            and set(node.get("required", [])) == set(node.get("properties", {})))

schema = {
    "type": "object",
    "properties": {"title": {"type": "string"}, "priority": {"type": "integer"}},
    "required": [],                 # (1)
    "additionalProperties": True,   # (2)
}

check("strict schema", is_strict(schema), True)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# schema = {
#     "type": "object",
#     "properties": {"title": {"type": "string"}, "priority": {"type": "integer"}},
#     "required": ["title", "priority"],
#     "additionalProperties": False,
# }


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Retry that feeds the error back
# ════════════════════════════════════════════════════════
# parse_with_retry(send, model_cls, max_attempts=3) -> (instance, attempts)
#   • send(feedback) -> raw text. feedback is None on the first attempt,
#     otherwise the text you want the model to act on
#   • parse with loads_or_none, then model_cls.model_validate
#   • on ValidationError, retry with validation_feedback(exc)
#   • when loads_or_none returns None, retry with
#     "Output was not valid JSON. Return a single JSON object only."
#   • after max_attempts, raise RuntimeError(f"gave up after {max_attempts} attempts")

SCRIPTED = [
    'sure, here: {"category": "billing", "severity": 9, "summary": "x"}',   # severity out of range
    'oops',                                                                # not JSON at all
    '{"category": "billing", "severity": 4, "summary": "double charge", "needs_human": true}',
]

def make_sender(scripted):
    seen = []
    def send(feedback):
        seen.append(feedback)
        return scripted[len(seen) - 1]
    return send, seen

def parse_with_retry(send, model_cls, max_attempts=3):
    pass  # to complete

send, seen = make_sender(SCRIPTED)
got = parse_with_retry(send, TriageResult)
check("validated result", lambda: got[0].model_dump(),
      {"category": "billing", "severity": 4, "summary": "double charge", "needs_human": True})
check("attempts used", lambda: got[1], 3)
check("attempt 2 saw the schema error", lambda: (seen[1] or "").splitlines()[0],
      "Your JSON failed validation:")
check("attempt 3 saw the parse error", lambda: seen[2],
      "Output was not valid JSON. Return a single JSON object only.")

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def parse_with_retry(send, model_cls, max_attempts=3):
#     feedback = None
#     for attempt in range(1, max_attempts + 1):
#         data = loads_or_none(send(feedback))
#         if data is None:
#             feedback = "Output was not valid JSON. Return a single JSON object only."
#             continue
#         try:
#             return model_cls.model_validate(data), attempt
#         except ValidationError as exc:
#             feedback = validation_feedback(exc)
#     raise RuntimeError(f"gave up after {max_attempts} attempts")


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# loads_or_none takes the OUTERMOST braces, which breaks on trailing prose that
# contains a brace. repair_json(text) -> dict | None does it properly:
#   1. strip a markdown code fence
#   2. extract the FIRST BALANCED {...}, counting braces while ignoring any
#      inside strings, honouring backslash escapes
#   3. drop trailing commas before } or ]
#   4. parse, returning None on failure and never raising
# A truncated object that never closes must return None, not a partial dict.

def repair_json(text):
    pass  # to complete

CASES = [
    '{"a": 1, "b": [2, 3]}',
    '```json\n{"a": 1}\n```',
    'Here you go: {"a": 1, "note": "use {braces} freely"} done!',
    '{"a": 1, "b": [2, 3,],}',
    '{"a": 1, "b": ',            # truncated, never closed
    'no json at all',
]

# One check over all six, so a stub that returns None cannot pass the None cases.
check("six cases", lambda: [repair_json(t) for t in CASES], [
    {"a": 1, "b": [2, 3]},
    {"a": 1},
    {"a": 1, "note": "use {braces} freely"},
    {"a": 1, "b": [2, 3]},
    None,
    None,
])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def _first_object(text):
#     start = text.find("{")
#     if start == -1:
#         return None
#     depth, in_string, escaped = 0, False, False
#     for i in range(start, len(text)):
#         ch = text[i]
#         if in_string:
#             if escaped:
#                 escaped = False
#             elif ch == "\\":
#                 escaped = True
#             elif ch == '"':
#                 in_string = False
#             continue
#         if ch == '"':
#             in_string = True
#         elif ch == "{":
#             depth += 1
#         elif ch == "}":
#             depth -= 1
#             if depth == 0:
#                 return text[start:i + 1]
#     return None                      # never closed, so truncated
#
# def repair_json(text):
#     cleaned = re.sub(r"```(?:json)?", "", text).strip()
#     candidate = _first_object(cleaned)
#     if candidate is None:
#         return None
#     candidate = re.sub(r",(\s*[}\]])", r"\1", candidate)   # trailing commas
#     try:
#         return json.loads(candidate)
#     except json.JSONDecodeError:
#         return None


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Pydantic emits a schema the API will not accept as strict. Fix it in general.
# to_strict_schema(model_cls) -> dict that, starting from
# model_cls.model_json_schema() and without mutating it:
#   • on every node with "type" == "object", sets "additionalProperties" to False
#     and "required" to all its property names, in declaration order
#   • removes every "title" key anywhere (noise the API does not need)
#   • recurses through "$defs", "properties" values and "items"

class Address(BaseModel):
    city: str
    zipcode: str

class Customer(BaseModel):
    name: str
    address: Address
    tags: list[str]

def to_strict_schema(model_cls):
    pass  # to complete

got = to_strict_schema(Customer)
check("no titles left", lambda: "title" in json.dumps(got["properties"]), False)
check("root is strict", lambda: (got["additionalProperties"], got["required"]),
      (False, ["name", "address", "tags"]))
check("nested is strict", lambda: (got["$defs"]["Address"]["additionalProperties"],
                                   got["$defs"]["Address"]["required"]),
      (False, ["city", "zipcode"]))

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# import copy
# def _strictify(node):
#     if isinstance(node, list):
#         return [_strictify(x) for x in node]
#     if not isinstance(node, dict):
#         return node
#     out = {k: _strictify(v) for k, v in node.items() if k != "title"}
#     if out.get("type") == "object":
#         out["additionalProperties"] = False
#         out["required"] = list(out.get("properties", {}).keys())
#     return out
#
# def to_strict_schema(model_cls):
#     return _strictify(copy.deepcopy(model_cls.model_json_schema()))


---
## 1.6 Prompt caching

An agent resends its entire context every turn, so by turn fifteen you are re-uploading a hundred
thousand tokens in order to ask a two-hundred-token question. Caching lets the server keep the
prefix it already processed. For a long-running agent that is not an optimisation, it is the
difference between viable and not. The catch is that it fails silently, so this section is as much
about noticing when it stopped working as about switching it on.

In [ ]:
# ── Where the breakpoint goes ────────────────────────────────
request = {
    "model": MODEL,
    "max_tokens": DEMO_MAX_TOKENS,
    "tools": [],                                   # renders FIRST (position 0)
    "system": [
        {"type": "text", "text": "You are a contract analyst."},           # frozen
        {"type": "text", "text": "<50k tokens of policy documents>",
         "cache_control": {"type": "ephemeral"}},  # breakpoint: caches tools + system
    ],
    "messages": [{"role": "user", "content": "Does clause 7 permit sublicensing?"}],  # volatile
}
resp = call(**request)
if resp is not None:
    u = resp.usage
    print("write:", u.cache_creation_input_tokens,
          "| read:", u.cache_read_input_tokens,
          "| uncached:", u.input_tokens)


### Lesson: Prefix matching is the whole model

Remember the amnesiac consultant from 1.1, the one who needs the entire case file re-handed to
them every call. Prompt caching is the bookmark you leave in that file. As long as every page
before the bookmark is byte for byte what it was last time, they flip straight to it instead of
starting at page one. That is where the roughly 90% discount comes from.

Bookmarks are unforgiving. Scribble one word on page 3 and nothing after it can be trusted, so the
whole file gets read again. Hence *prefix* match: almost the same prompt buys you nothing, only
identical-up-to-here does.

The render order is fixed, `tools` then `system` then `messages`, so design the prompt with
stability decreasing down the page: frozen content first, volatile content last.


In [ ]:
# ── TTL, economics, and the minimum that actually caches ─────
# write 1.25x base input for the default 5-minute ttl, 2.00x for "1h"
# read  0.10x base input
# break-even: 2 requests at 5m ttl, 3 requests at 1h ttl
long_lived = {"type": "ephemeral", "ttl": "1h"}

MIN_CACHEABLE = {          # tokens; below this nothing caches, silently
    "claude-opus-5":    512,
    "claude-fable-5":   512,
    "claude-opus-4-8": 1024,
    "claude-sonnet-5": 1024,
    "claude-opus-4-7": 2048,
    "claude-opus-4-6": 4096,
    "claude-haiku-4-5": 4096,
}
for name, minimum in MIN_CACHEABLE.items():
    print(f"{name:<18} min cacheable prefix: {minimum:>5} tokens")

# Note it is not monotonic across generations. A prompt that cached fine on Opus 5
# silently stops caching the day a config change points it at Haiku. No error field,
# just cache_creation_input_tokens: 0 forever. Verify with usage, never assume.


### Lesson: Silent invalidators

Caching fails quietly. No warning, no error field, just a bill that never goes down. Every
invalidator is the same mistake in a different coat: something scribbled on a page before the
bookmark and nobody noticed.

A timestamp in the system prompt is a librarian stamping today's date on page 1 every time the
book opens, so the bookmark can never hold. Unsorted JSON is a library reshelving the same books in
a different order every night: identical contents, unrecognisable sequence. A per-user or
feature-flagged tool list sits at position 0, so it invalidates everything behind it. Conditional
system sections fork your cache into 2^n distinct prefixes.

The next cell renders one bad prefix twice and diffs it against a good one.


In [ ]:
# ── A prefix that can never be reused ────────────────────────
def build_prefix_bad(user):
    return (f"Today is {time.strftime('%Y-%m-%d %H:%M:%S')}. "     # changes every second
            f"User: {user['id']}. "                                # per-user fork
            f"Config: {json.dumps(user['prefs'])}")                # insertion-ordered keys

# ── Frozen prefix, volatile content pushed past the breakpoint ─
SYSTEM_FROZEN = "You are a contract analyst. Cite clause numbers."

def build_prefix_good(user):
    return f"{SYSTEM_FROZEN} Config: {json.dumps(user['prefs'], sort_keys=True)}"


same_prefs_other_order = {"id": "u_42", "prefs": {"verbosity": "low", "lang": "fr"}}
user                   = {"id": "u_42", "prefs": {"lang": "fr", "verbosity": "low"}}

print("bad  identical across calls:", build_prefix_bad(user) == build_prefix_bad(user))
print("good identical across calls:", build_prefix_good(user) == build_prefix_good(user))
print("good survives key reorder  :",
      build_prefix_good(user) == build_prefix_good(same_prefs_other_order))


### Lesson: What each change actually costs you

Not every edit burns everything. Changing `tool_choice`, toggling thinking, or attaching images
invalidates only the messages tier. Changing the system prompt keeps the tools tier. Changing tools
or the model rebuilds all three, because tools render at position 0.

Two subtler failures are worth knowing before they bite. **Concurrent fan-out**: N identical
requests fired at the same instant all miss, because an entry becomes readable only once the first
response starts streaming. Send one, wait for the first token, then fire the rest. **The 20-block
lookback**: a breakpoint searches backwards at most 20 content blocks, so an agent turn appending
30 tool_use and tool_result blocks loses the trail. Add an intermediate breakpoint every 15 blocks
or so.

And there are two escape hatches: a mid-conversation `{"role": "system"}` message rather than
editing top-level `system`, and `tool_addition` / `tool_removal` blocks rather than editing `tools`.


In [ ]:
# ── The invalidation hierarchy ───────────────────────────────
#   change                          tools   system  messages
#   tool definitions / model         x       x        x       (full rebuild)
#   system prompt content            ok      x        x
#   tool_choice / thinking / images  ok      ok       x
#   new message content              ok      ok       x       (expected, harmless)

# ── Pre-warming: pay the write before traffic arrives ────────
prewarm = {
    "model": MODEL,
    "max_tokens": 0,                     # prefill only: content=[], no output billed
    "system": [{"type": "text", "text": SYSTEM_FROZEN,
                "cache_control": {"type": "ephemeral"}}],
    "messages": [{"role": "user", "content": "warmup"}],
}
# Worth it when first-request latency is user-visible AND the prefix is large AND
# there is a quiet moment before traffic (startup, deploy). Not for steady traffic.
print(json.dumps(prewarm, indent=1)[:180])


### 🏋️ Exercises 1.6


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 1 (warm-up) — Which models will actually cache this?
# ════════════════════════════════════════════════════════
# Your cacheable prefix is 800 tokens. Using MIN_CACHEABLE from the lesson,
# build the sorted list of models where it caches at all. One expression.

CACHES_ON = []

check("models that cache an 800-token prefix", CACHES_ON,
      ["claude-fable-5", "claude-opus-5"])

# ════════════════════════════════════════════════════════
# SOLUTION (uncomment to check)
# ════════════════════════════════════════════════════════
# CACHES_ON = sorted(m for m, minimum in MIN_CACHEABLE.items() if minimum <= 800)
#   Everywhere else it silently does not cache. There is no error to catch,
#   only cache_creation_input_tokens staying at 0 forever.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 2 (fill in the gaps) — Make this prefix cacheable
# ════════════════════════════════════════════════════════
# build_prefix currently defeats the cache twice over. Fix the two marked
# lines so the rendered prefix is byte-identical across calls and unaffected
# by the insertion order of `prefs`.

def build_prefix(user):
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")        # (1) this has no business in a prefix
    return f"{SYSTEM_FROZEN} [{stamp}] Config: {json.dumps(user['prefs'])}"   # (2) key order

a = {"id": "u_42", "prefs": {"lang": "fr", "verbosity": "low"}}
b = {"id": "u_99", "prefs": {"verbosity": "low", "lang": "fr"}}

check("no timestamp in the prefix", any(ch.isdigit() for ch in build_prefix(a)), False)
check("stable across key order", build_prefix(a) == build_prefix(b), True)

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def build_prefix(user):
#     return f"{SYSTEM_FROZEN} Config: {json.dumps(user['prefs'], sort_keys=True)}"
#   The timestamp goes in the user turn, after the breakpoint, if you need it at all.


In [ ]:
# ════════════════════════════════════════════════════════
# EXERCISE 3 — Place breakpoints at stability boundaries
# ════════════════════════════════════════════════════════
# plan_breakpoints(segments, min_cacheable, max_breakpoints=4) -> list of the
# segment names AFTER which to attach cache_control. `segments` are in render
# order, each {"name", "tokens", "stability"}. Rules:
#   • a candidate is the LAST segment of each run of equal stability
#   • never break on a "per_request" run, it is never reused
#   • a candidate only counts if the cumulative tokens up to and including it
#     reach min_cacheable
#   • at most max_breakpoints per request: keep the LAST ones, the deepest prefixes

SEGMENTS = [
    {"name": "tools",          "tokens":  900, "stability": "static"},
    {"name": "system_core",    "tokens": 1500, "stability": "static"},
    {"name": "persona",        "tokens":  300, "stability": "per_session"},
    {"name": "retrieved_docs", "tokens": 4000, "stability": "per_session"},
    {"name": "history",        "tokens": 2500, "stability": "per_turn"},
    {"name": "question",       "tokens":   80, "stability": "per_request"},
]

def plan_breakpoints(segments, min_cacheable, max_breakpoints=4):
    pass  # to complete

check("Opus 5 (512)",  plan_breakpoints(SEGMENTS, 512),
      ["system_core", "retrieved_docs", "history"])
check("Haiku 4.5 (4096)", plan_breakpoints(SEGMENTS, 4096),
      ["retrieved_docs", "history"])

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def plan_breakpoints(segments, min_cacheable, max_breakpoints=4):
#     candidates, cumulative = [], 0
#     for i, seg in enumerate(segments):
#         cumulative += seg["tokens"]
#         last_of_run = (i == len(segments) - 1
#                        or segments[i + 1]["stability"] != seg["stability"])
#         if not last_of_run or seg["stability"] == "per_request":
#             continue
#         if cumulative >= min_cacheable:
#             candidates.append(seg["name"])
#     return candidates[-max_breakpoints:]


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Does caching pay off for this workload?
# cache_economics(n, prefix_tokens, suffix_tokens, output_tokens, price_in,
#                 price_out, ttl="5m") -> {"uncached", "cached", "saved",
#                 "break_even"}, dollars rounded to 6 decimals.
#   • uncached: every request pays (prefix + suffix) at price_in, output at price_out
#   • cached  : request 1 writes the prefix at price_in * write_mult (1.25 for "5m",
#               2.00 for "1h"); requests 2..n read it at price_in * 0.10; suffix and
#               output always cost full price
#   • break_even: the smallest request count at which cached < uncached (may exceed n)
# Prices are $ per 1M tokens.

def cache_economics(n, prefix_tokens, suffix_tokens, output_tokens,
                    price_in, price_out, ttl="5m"):
    pass  # to complete

check("5m", cache_economics(10, 50_000, 500, 400, 5.00, 25.00),
      {"uncached": 2.625, "cached": 0.6625, "saved": 1.9625, "break_even": 2})
check("1h", cache_economics(10, 50_000, 500, 400, 5.00, 25.00, ttl="1h"),
      {"uncached": 2.625, "cached": 0.85, "saved": 1.775, "break_even": 3})

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def cache_economics(n, prefix_tokens, suffix_tokens, output_tokens,
#                     price_in, price_out, ttl="5m"):
#     write_mult = {"5m": 1.25, "1h": 2.00}[ttl]
#
#     def uncached_cost(k):
#         return (k * (prefix_tokens + suffix_tokens) * price_in
#                 + k * output_tokens * price_out) / 1_000_000
#
#     def cached_cost(k):
#         prefix = prefix_tokens * price_in * (write_mult + 0.10 * (k - 1))
#         return (prefix + k * suffix_tokens * price_in
#                 + k * output_tokens * price_out) / 1_000_000
#
#     break_even = next((k for k in range(1, 1000) if cached_cost(k) < uncached_cost(k)), None)
#     return {"uncached": round(uncached_cost(n), 6),
#             "cached": round(cached_cost(n), 6),
#             "saved": round(uncached_cost(n) - cached_cost(n), 6),
#             "break_even": break_even}


In [ ]:
# ════════════════════════════════════════════════════════
# OPTIONAL STRETCH — off the main path, skip on a first read
# ════════════════════════════════════════════════════════
# Two consecutive requests, and the bill went up. Which tier died?
# The rendered prefix is tools, then system, then messages.
# diagnose(req_a, req_b) -> (tier, offset) where
#   • render(req) = json.dumps(req["tools"]) + json.dumps(req["system"])
#                                            + json.dumps(req["messages"])
#   • offset is the index of the first differing character, or the length of
#     the shorter render when one is a prefix of the other, None if identical
#   • tier is which section that offset lands in, or "none" when identical

base = {
    "tools": [{"name": "search", "description": "search docs"}],
    "system": [{"type": "text", "text": "You are an analyst."}],
    "messages": [{"role": "user", "content": "clause 7?"}],
}
changed_msg    = {**base, "messages": [{"role": "user", "content": "clause 9?"}]}
changed_system = {**base, "system": [{"type": "text", "text": "You are an auditor."}]}
changed_tools  = {**base, "tools": [{"name": "search", "description": "search all docs"}]}

def diagnose(req_a, req_b):
    pass  # to complete

check("identical",  diagnose(base, base),           ("none", None))
check("message",    diagnose(base, changed_msg),    ("messages", 136))
check("system",     diagnose(base, changed_system), ("system", 89))
check("tools",      diagnose(base, changed_tools),  ("tools", 43))

# ════════════════════════════════════════════════════════
# SOLUTION
# ════════════════════════════════════════════════════════
# def diagnose(req_a, req_b):
#     def sections(req):
#         return (json.dumps(req["tools"]), json.dumps(req["system"]),
#                 json.dumps(req["messages"]))
#     a_parts, b_parts = sections(req_a), sections(req_b)
#     a, b = "".join(a_parts), "".join(b_parts)
#     if a == b:
#         return "none", None
#     offset = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
#     bounds = [("tools", len(a_parts[0])),
#               ("system", len(a_parts[0]) + len(a_parts[1])),
#               ("messages", len(a))]
#     return next(name for name, end in bounds if offset < end), offset


---
## 📋 Module 1 Recap

**The main path.** What you should be able to do without looking anything up.

| Section | Topic | Test |
|---|---|---|
| 1.1 | Roles and authority | Keep untrusted data in `user`, operator instructions in `system` |
| 1.1 | Content blocks | Read a response without assuming `content[0].text` |
| 1.1 | Transcript rules | `messages[0]` is `user`; consecutive same-role turns merge |
| 1.1 | Multi-turn state | Append `response.content` whole, so thinking and tool_use blocks survive |
| 1.2 | Sampling mechanics | Explain what temperature does to a distribution, and why the knob is gone |
| 1.2 | Effort and thinking | Map a workload onto `low` through `max`; adaptive thinking, not `budget_tokens` |
| 1.2 | Stop handling | Branch on `stop_reason` before parsing; know which values mean "incomplete" |
| 1.3 | Token counting | `count_tokens` per model, never `tiktoken`, never `chars / 4` |
| 1.3 | Usage accounting | Prompt size is `input` + `cache_creation` + `cache_read`, and cost follows from it |
| 1.3 | Context budgeting | Trim history to a budget without orphaning a turn |
| 1.4 | Stream protocol | One accumulator per block index; rebuild the content list from raw events |
| 1.4 | Partial JSON | Accumulate `input_json_delta`, parse only at `content_block_stop` |
| 1.5 | Structured output | `messages.parse` with Pydantic; a strict schema needs `additionalProperties: false` |
| 1.5 | Defensive parsing | Turn "probably JSON" into an object or a clean `None` |
| 1.5 | Validation retry | Feed the validation error back, with a capped attempt count |
| 1.6 | Cache prefix model | Byte-identical prefix, stability decreasing down the page |
| 1.6 | Silent invalidators | Timestamps, unsorted JSON, per-user tool lists |
| 1.6 | Breakpoint placement | At stability boundaries, at most 4, above the model's minimum |

**Optional stretch.** Harder drills sitting off the main path. Worth doing, in no particular
hurry, and none of them is a prerequisite for module 2.

| Section | Drill |
|---|---|
| 1.1 | Placement rules for a mid-conversation `role: "system"` message |
| 1.2 | Nucleus (top-p) sampling from scratch |
| 1.2 | A gateway shim that rewrites legacy params instead of eating a 400 |
| 1.3 | Token-aware chunking with overlap and an injected counter |
| 1.3 | A full run cost ledger across cache tiers and TTLs |
| 1.4 | Stream latency metrics: TTFT, generation time, tokens/s, worst stall |
| 1.4 | Full message reconstruction, not just the content list |
| 1.5 | `repair_json`: balanced-brace extraction that survives braces inside strings |
| 1.5 | A recursive Pydantic to strict-JSON-schema converter |
| 1.6 | Cache economics and the break-even request count |
| 1.6 | Locate the first divergent byte and name the cache tier it destroyed |

**Next:** `module2.ipynb` covers tool definitions, the `tool_use` to `tool_result` loop, parallel
calls, `tool_choice`, and feeding tool failures back to the model.
